# Portfolio optimization

**Prerequisites:** notebook **05** (portfolio construction and valuation).

This notebook demonstrates the LP-based portfolio optimizer. The optimizer
takes a portfolio of bonds, an objective (maximize yield), and constraints
(weight bounds, rating caps, turnover limits), then solves for optimal
weights and generates a trade list.

Two entry points are shown:

1. **`optimize_max_yield`** — convenience helper that maximizes value-weighted
   YTM subject to a CCC exposure cap.
2. **`optimize_portfolio`** — general optimizer that accepts a full
   `PortfolioOptimizationSpec` JSON with arbitrary objective and constraints.

In [1]:
import json
from datetime import date

import pandas as pd

from finstack.core.market_data import DiscountCurve, MarketContext
from finstack.portfolio import (
    optimize_max_yield,
    optimize_portfolio,
    value_portfolio,
)

print("Imports OK")

Imports OK


## Market context

A single USD OIS discount curve used by all bonds in the portfolio.

In [2]:
mc = MarketContext()
curve = DiscountCurve(
    "USD-OIS",
    date(2025, 1, 15),
    [
        (0.0, 1.0),
        (0.25, 0.9875),
        (0.5, 0.975),
        (1.0, 0.95),
        (2.0, 0.90),
        (5.0, 0.75),
        (10.0, 0.55),
    ],
    day_count="act_365f",
)
mc.insert(curve)
market_json = mc.to_json()
print("Market context ready")

Market context ready


## Portfolio: five fixed-rate bonds

Each bond has a `rating` tag so we can apply tag-based constraints in the
optimizer. Coupons range from 3% (AAA) to 8% (CCC), reflecting typical
credit spread tiers.

In [3]:
def make_bond_position(pos_id, bond_id, coupon, maturity, rating, notional=1_000_000):
    """Build a portfolio position dict for a fixed-rate bond."""
    return {
        "position_id": pos_id,
        "entity_id": "FUND",
        "instrument_id": bond_id,
        "instrument_spec": {
            "type": "bond",
            "spec": {
                "id": bond_id,
                "notional": {"amount": str(notional), "currency": "USD"},
                "issue_date": "2024-01-15",
                "maturity": maturity,
                "discount_curve_id": "USD-OIS",
                "cashflow_spec": {
                    "Fixed": {
                        "coupon_type": "Cash",
                        "rate": coupon,
                        "freq": {"count": 6, "unit": "months"},
                        "dc": "Thirty360",
                        "bdc": "following",
                        "calendar_id": "weekends_only",
                        "end_of_month": False,
                        "payment_lag_days": 0,
                        "stub": "None",
                    }
                },
                "credit_curve_id": None,
                "call_put": None,
                "attributes": {"tags": [], "meta": {}},
                "pricing_overrides": {},
            },
        },
        "quantity": 1.0,
        "unit": "units",
        "tags": {"rating": rating, "sector": "corporate"},
    }


positions = [
    make_bond_position("POS-AAA", "BOND-AAA-5Y", 0.03, "2029-01-15", "AAA"),
    make_bond_position("POS-AA",  "BOND-AA-3Y",  0.04, "2027-01-15", "AA"),
    make_bond_position("POS-BBB", "BOND-BBB-5Y", 0.055, "2029-01-15", "BBB"),
    make_bond_position("POS-BB",  "BOND-BB-7Y",  0.07, "2031-01-15", "BB"),
    make_bond_position("POS-CCC", "BOND-CCC-5Y", 0.08, "2029-01-15", "CCC"),
]

portfolio_spec = json.dumps({
    "id": "bond-fund",
    "as_of": "2025-01-15",
    "base_ccy": "USD",
    "entities": {"FUND": {"id": "FUND"}},
    "positions": positions,
})

print(f"Portfolio: {len(positions)} bonds")
for p in positions:
    print(f"  {p['position_id']:10s}  coupon={p['instrument_spec']['spec']['cashflow_spec']['Fixed']['rate']:.1%}  rating={p['tags']['rating']}")

Portfolio: 5 bonds
  POS-AAA     coupon=3.0%  rating=AAA
  POS-AA      coupon=4.0%  rating=AA
  POS-BBB     coupon=5.5%  rating=BBB
  POS-BB      coupon=7.0%  rating=BB
  POS-CCC     coupon=8.0%  rating=CCC


## Baseline valuation

Value the portfolio before optimization to see the starting state.

In [4]:
val = json.loads(value_portfolio(portfolio_spec, market_json))

rows = []
for pos_id, pv in val.get("position_values", {}).items():
    rows.append({
        "position": pv["position_id"],
        "pv": float(pv["value_base"]["amount"]),
        "currency": pv["value_base"]["currency"],
    })

df_val = pd.DataFrame(rows)
total = val["total_base_ccy"]
print(f"Total portfolio value: {float(total['amount']):,.2f} {total['currency']}")
df_val

Total portfolio value: 5,148,630.66 USD


,position,pv,currency
0,POS-AAA,9.193476e+05,USD
1,POS-AA,9.950086e+05,USD
2,POS-BBB,1.020541e+06,USD
3,POS-BB,1.092000e+06,USD
4,POS-CCC,1.121734e+06,USD


## 1. Quick optimization: maximize yield with CCC cap

`optimize_max_yield` maximizes value-weighted YTM while constraining CCC
exposure (positions tagged `rating="CCC"`) to a specified limit.

The optimizer uses an LP formulation: weights are value-based shares that
sum to 1.0 (fully invested budget constraint). The objective and constraints
are linearized in terms of these weights.

In [5]:
result = json.loads(optimize_max_yield(
    portfolio_spec, market_json, ccc_limit=0.15, strict_risk=False
))

print(f"Status:          {result['status_label']}")
print(f"Objective (YTM): {result['objective_value']:.4%}")
print(f"CCC weight:      {result['ccc_weight']:.2%}")
print()

weights_df = pd.DataFrame([
    {
        "position": pos_id,
        "current_wt": result["current_weights"].get(pos_id, 0),
        "optimal_wt": result["optimal_weights"].get(pos_id, 0),
        "delta": result["weight_deltas"].get(pos_id, 0),
    }
    for pos_id in result["optimal_weights"]
])
weights_df

Status:          Optimal
Objective (YTM): 5.2623%
CCC weight:      0.00%



,position,current_wt,optimal_wt,delta
0,POS-AAA,0.178562,1.0,0.821438
1,POS-AA,0.193257,0.0,-0.193257
2,POS-BBB,0.198216,0.0,-0.198216
3,POS-BB,0.212095,0.0,-0.212095
4,POS-CCC,0.217870,0.0,-0.217870


## 2. General optimizer: custom objective and constraints

`optimize_portfolio` accepts a `PortfolioOptimizationSpec` that lets you
define any linear objective and mix of constraints:

- **Objective**: `Maximize` or `Minimize` a `MetricExpr`
  - `WeightedSum` / `ValueWeightedAverage` of a `PerPositionMetric`
  - `TagExposureShare` for tag-based exposure
- **Constraints**:
  - `TagExposureLimit` / `TagExposureMinimum` — cap/floor on tag buckets
  - `WeightBounds` — min/max weight per position (with filters)
  - `MaxTurnover` — total portfolio turnover limit
  - `Budget` — weight normalization (default: sum to 1.0)
  - `MetricBound` — bound on any metric expression

Below we maximize YTM subject to:
- CCC weight ≤ 10%
- BB weight ≤ 30%
- Max turnover ≤ 40%
- Each position weight in [0%, 40%]

In [6]:
opt_spec = json.dumps({
    "portfolio": json.loads(portfolio_spec),
    "objective": {
        "Maximize": {
            "ValueWeightedAverage": {
                "metric": {"Metric": "Ytm"}
            }
        }
    },
    "constraints": [
        {"TagExposureLimit": {
            "label": "ccc_cap",
            "tag_key": "rating",
            "tag_value": "CCC",
            "max_share": 0.10,
        }},
        {"TagExposureLimit": {
            "label": "bb_cap",
            "tag_key": "rating",
            "tag_value": "BB",
            "max_share": 0.30,
        }},
        {"MaxTurnover": {
            "label": "turnover_limit",
            "max_turnover": 0.40,
        }},
        {"WeightBounds": {
            "label": "position_limits",
            "filter": "All",
            "min": 0.0,
            "max": 0.40,
        }},
    ],
    "weighting": "ValueWeight",
    "missing_metric_policy": "Zero",
    "label": "max_yield_constrained",
})

result = json.loads(optimize_portfolio(opt_spec, market_json))

print(f"Status:    {result['status_label']}")
print(f"Feasible:  {result['is_feasible']}")
print(f"Objective: {result['objective_value']:.4%}")
print(f"Turnover:  {result['turnover']:.2%}")
print(f"Binding:   {result['binding_constraints']}")

Status:    Optimal
Feasible:  True
Objective: 0.0000%
Turnover:  40.00%
Binding:   ['budget', 'ccc_cap']


### Optimal weights and deltas

In [7]:
weights_df = pd.DataFrame([
    {
        "position": pos_id,
        "current_wt": result["current_weights"].get(pos_id, 0),
        "optimal_wt": wt,
        "delta": result["weight_deltas"].get(pos_id, 0),
    }
    for pos_id, wt in result["optimal_weights"].items()
])
weights_df.style.format({"current_wt": "{:.2%}", "optimal_wt": "{:.2%}", "delta": "{:+.2%}"})

,position,current_wt,optimal_wt,delta
0,POS-AAA,17.86%,17.86%,+0.00%
1,POS-AA,19.33%,39.33%,+20.00%
2,POS-BBB,19.82%,11.61%,-8.21%
3,POS-BB,21.21%,21.21%,+0.00%
4,POS-CCC,21.79%,10.00%,-11.79%


### Trade list

The optimizer returns a sorted trade list with direction, quantities, and
weight changes.

In [8]:
if result["trades"]:
    trades_df = pd.DataFrame(result["trades"])
    display(trades_df)
else:
    print("No trades needed (portfolio already optimal).")

,position_id,instrument_id,trade_type,current_quantity,target_quantity,delta_quantity,direction,current_weight,target_weight
0,POS-AA,BOND-AA-3Y,Existing,1.0,2.034892,1.034892,Buy,0.193257,0.393257
1,POS-CCC,BOND-CCC-5Y,Existing,1.0,0.458989,-0.541011,Sell,0.217870,0.100000
2,POS-BBB,BOND-BBB-5Y,Existing,1.0,0.585655,-0.414345,Sell,0.198216,0.116086


### Constraint diagnostics

Dual values (shadow prices) show how much the objective would improve per
unit relaxation of each constraint. Slack values show how far each
constraint is from binding.

In [9]:
if result["dual_values"]:
    diag_df = pd.DataFrame([
        {
            "constraint": name,
            "dual_value": result["dual_values"].get(name, 0),
            "slack": result["constraint_slacks"].get(name, 0),
            "binding": name in result["binding_constraints"],
        }
        for name in result["constraint_slacks"]
    ])
    display(diag_df)
else:
    print("No constraint diagnostics available.")

No constraint diagnostics available.


## Summary

| Function | Use case |
|---|---|
| `optimize_max_yield` | Quick yield maximization with CCC cap |
| `optimize_portfolio` | General LP optimizer with custom objective + constraints |

The optimizer runs entirely in Rust and returns JSON results including
optimal weights, trade lists, dual values, and binding constraint
diagnostics. All constraint types are composable and can be mixed freely.